In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, OrdinalEncoder)
pd.set_option('display.max_columns', 1000)


In [2]:
df=pd.read_csv("/home/salah-mohammed/Data_Science/ds_jobs_project/data/cleaned_dfs/big_updated_df.csv")
# df=df.drop(columns=['Unnamed: 0'	,'Rating'	,'Company Name'	,'Headquarters'	,'Size'	,'Founded'	,'Type of ownership'	,'Industry'	,'Sector',	'Revenue'	,'Competitors'])
df.head()

,Job Title,Company,Location,Salary Estimate,Job Description,Job Age,Unnamed: 0,Rating,Company Name,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Country,Hourly,Glassdoor_Estimate,Employer_Provided,currency,min_salary,max_salary,avg_salary,Skills,cleaned_Job,Writing skills,SAS,SAP,Relational databases,Data analytics,Project coordination,SDLC,Mandarin,Quantitative analysis,Forecasting,Public speaking,Systems design,Tooling,Organization design,Application development,SharePoint,SolidWorks,C++,Data management,Banking,TestNG,Tableau,Authentication,SVN,Public health,Web analytics,Cassandra,Yardi,IIS,JavaScript,Predictive analytics,Natural language processing,Web accessibility,Quality assurance,Virtualization,Time management,Research & development,Conflict management,Business analysis,Computer science,Embedded software,Debugging,CI/CD,Computer vision,Moving,E-commerce,Regression analysis,Microsoft Access,Oracle,CAD,Business processes,Data mining,System design,Data collection,Salesforce Marketing Cloud,Fiddler,Ansible,D3.js,Multilingual,Pro/ENGINEER,T-SQL,Waterfall,C#,Wireframing,Ontology,Military,XML,Software troubleshooting,SSRS,Market analysis,Epic,OOP,Marketing,CRM software,WAN,Project management,SASS,Risk management,Business requirements,PaaS,HP ALM,Recruiting,Performance tuning,Python,Web development,Program management,Microbiology,Customer retention,Statistical analysis,FedRAMP,Microservices,Release management,Database management,Solution architecture,Controlling experience,Vendor management,Django,Node.js,TypeScript,Elasticsearch,Linux,Distributed systems,Schematics,Databases,Mentoring,Jira,Scala,5G,Wealth management,Help desk,REST,SOX,Spark,Warehouse management system,Google Cloud Platform,Sales,Internet of things,Filing,Logistics,IT,Systems engineering,Microsoft PowerPoint,Financial analysis,.NET,Robotics,Signal processing,QlikView,Continuous improvement,Investment,Data centre experience,Big data,AWS,DevOps,Data structures,Software architecture,Cost management,Scripting,NetSuite,Requirements gathering,Requirements analysis,Data entry,Visual Basic,Load balancing,Microsoft Word,Test automation,MATLAB,Scalability,Financial modeling,Salesforce,Apache Hive,Redshift,Google Suite,Change management,SAP SuccessFactors,Operations management,Data visualization,FISMA,TCP,Confluence,SDKs,Russian,R,Data analysis skills,Deployment,Product development,Computer skills,Salt,Windows,Organisational skills,Management,Kanban,Analysis skills,UI,Performance management,Microsoft Powerpoint,Software testing,Process improvement,English,Figma,Pivot tables,Mechanical engineering,Trello,Medical imaging,Statistical software,IT service management,Hospitality,Cloud development,SaaS,System architecture,VoIP,GAMP,Machine learning,Presentation skills,Microsoft Office,Data warehouse,Disaster recovery,Front-end development,DoD experience,RBAC,ERP systems,System administration,Analytics,Maths,Azure,A/B testing,Japanese,Market research,Clinical development,Database development,Fraud prevention and detection,Bootstrap,TensorFlow,Test-driven development,Operating systems,New Relic,Usability,Math,Software development,Statistics,MongoDB,Contract management,.NET Core,Agile,Operations research,Content development,Marketing mix modeling,Terrafrom,Experimental design,Alteryx,Calculus,ServiceNow,Fanuc,Interviewing,Unity,Pricing,Computer networking,Customer service,Graph databases,FEA,French,DB2,GitHub,Bilingual,Mobile applications,SSO,Docker,Project management software,Data modelling,Microsoft SQL Server,Organizational skills,Clinical research,Supply chain,Taxonomy,HIPAA,Spanish,Cucumber (software testing tool),Telecommunication,Internet of Things,Visual Studio,Financial planning,Crystal Reports,Clinical laboratory experience,Data science,Quality control,EDI,NoSQL,Administrative experience,Biotechnology,Managed care,Team management,Construction,Unreal Engine,Test management tools,S3,Relationship management,Knowledge management,CSS,Root cau

### EDA Functions

In [3]:
"""UNIVARIATE PLOTTING FUNCTIONS FOR EDA"""
# Add the print statements to the function
def explore_categorical(df, x, fillna = True, placeholder = 'MISSING',
                        figsize = (6,4), order = None):
  """Creates a seaborn countplot with the option to temporarily fill missing values
  Prints statements about null values, cardinality, and checks for
  constant/quasi-constant features.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Make a copy of the dataframe and fillna
  temp_df = df.copy()
  # Before filling nulls, save null value counts and percent for printing
  null_count = temp_df[x].isna().sum()
  null_perc = null_count/len(temp_df)* 100
  # fillna with placeholder
  if fillna == True:
    temp_df[x] = temp_df[x].fillna(placeholder)
  # Create figure with desired figsize
  fig, ax = plt.subplots(figsize=figsize)
  # Plotting a count plot
  sns.countplot(data=temp_df, x=x, ax=ax, order=order)
  # Rotate Tick Labels for long names
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
  # Add a title with the feature name included
  ax.set_title(f"Column: {x}", fontweight='bold')

  # Fix layout and show plot (before print statements)
  fig.tight_layout()
  plt.show()
  print("*****"*10)
  # Print null value info
  print(f"- NaN's Found: {null_count} ({round(null_perc,2)}%)")
  # Print cardinality info
  nunique = temp_df[x].nunique()
  print(f"- Unique Values: {nunique}")
  # First find value counts of feature
  val_counts = temp_df[x].value_counts(dropna=False)
  # Define the most common value
  most_common_val = val_counts.index[0]
  # Define the frequency of the most common value
  freq = val_counts.values[0]
  # Calculate the percentage of the most common value
  perc_most_common = freq / len(temp_df) * 100
  # Print the results
  print(f"- Most common value: '{most_common_val}' occurs {freq} times ({round(perc_most_common,2)}%)")
  # print message if quasi-constant or constant (most common val more than 98% of data)
  if perc_most_common > 98:
    print(f"\n- [!] Warning: '{x}' is a constant or quasi-constant feature and should be dropped.")
  else:
    print("- Not constant or quasi-constant.")
  return fig, ax


# TO DO: add the new print statements from explore_categorical
def explore_numeric(df, x, figsize=(6,5) ):
  """Creates a seaborn histplot and boxplot with a share x-axis,
  Prints statements about null values, cardinality, and checks for
  constant/quasi-constant features.
  Source:{PASTE IN FINAL LESSON LINK}
  """

  ## Save null value counts and percent for printing
  null_count = df[x].isna().sum()
  null_perc = null_count/len(df)* 100


  ## Making our figure with gridspec for subplots
  gridspec = {'height_ratios':[0.7,0.3]}
  fig, axes = plt.subplots(nrows=2, figsize=figsize,
                           sharex=True, gridspec_kw=gridspec)
  # Histogram on Top
  sns.histplot(data=df, x=x, ax=axes[0])

  # Boxplot on Bottom
  sns.boxplot(data=df, x=x, ax=axes[1])

  ## Adding a title
  axes[0].set_title(f"Column: {x}", fontweight='bold')

  ## Adjusting subplots to best fill Figure
  fig.tight_layout()

  # Ensure plot is shown before message
  plt.show()

  print("*****"*10)
  # Print null value info
  print(f"- NaN's Found: {null_count} ({round(null_perc,2)}%)")
  # Print cardinality info
  nunique = df[x].nunique()
  print(f"- Unique Values: {nunique}")


  # Get the most most common value, its count as # and as %
  most_common_val_count = df[x].value_counts(dropna=False).head(1)
  most_common_val = most_common_val_count.index[0]
  freq = most_common_val_count.values[0]
  perc_most_common = freq / len(df) * 100

  print(f"- Most common value: '{most_common_val}' occurs {freq} times ({round(perc_most_common,2)}%)")

  # print message if quasi-constant or constant (most common val more than 98% of data)
  if perc_most_common > 98:
    print(f"\n- [!] Warning: '{x}' is a constant or quasi-constant feature and should be dropped.")
  else:
    print("- Not constant or quasi-constant.")
  return fig, axes


In [4]:
"""MULTIVARIATE PLOTTING FUNCTIONS VS. NUMERIC TARGET"""

def plot_categorical_vs_target(df, x, y='rating',figsize=(6,4),
                            fillna = True, placeholder = 'MISSING',
                            order = None):
  """Plots a combination of a seaborn barplot of means combined with
  a seaborn stripplot to show the spread of the data.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Make a copy of the dataframe and fillna
  temp_df = df.copy()
  # fillna with placeholder
  if fillna == True:
    temp_df[x] = temp_df[x].fillna(placeholder)

  # or drop nulls prevent unwanted 'nan' group in stripplot
  else:
    temp_df = temp_df.dropna(subset=[x])
  # Create the figure and subplots
  fig, ax = plt.subplots(figsize=figsize)

    # Barplot
  sns.barplot(data=temp_df, x=x, y=y, ax=ax, order=order, alpha=0.6,
              linewidth=1, edgecolor='black', errorbar=None)

  # Boxplot
  sns.stripplot(data=temp_df, x=x, y=y, hue=x, ax=ax,
                order=order, hue_order=order, legend=False,
                edgecolor='white', linewidth=0.5,
                size=3,zorder=0)
  # Rotate xlabels
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

  # Add a title
  ax.set_title(f"{x} vs. {y}", fontweight='bold')
  fig.tight_layout()
  return fig, ax


def plot_numeric_vs_target(df, x, y,
                           figsize=(6,4),
                           ):
  """Plots a seaborn regplot with Pearson's correlation (r) added
  to the title.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Calculate the correlation
  corr = df[[x,y]].corr().round(2)
  r = corr.loc[x,y]

  # Plot the data
  fig, ax = plt.subplots(figsize=figsize)
  scatter_kws={'ec':'white','alpha':0.8}
  sns.regplot(data=df, x=x, y=y, ax=ax,scatter_kws=scatter_kws)

  ## Add the title with the correlation
  ax.set_title(f"{x} vs. {y} (r = {r})", fontweight='bold')

  # Make sure the plot is shown before the print statement
  plt.show()

  return fig, ax

# + Cont'd Data Preperation phase

In [5]:
drop_cols=[ 'Job Title','Company', 'Location', 'Salary Estimate', "Job Description", 'Job Age' ,'Country','Skills','min_salary','max_salary','avg_salary','currency' ,'Unnamed: 0'	,'Rating'	,'Company Name'	,'Headquarters'	,'Size'	,'Founded'	,'Type of ownership'	,'Industry'	,'Sector',	'Revenue'	,'Competitors','cleaned_Job']

# drop title, company, location, job descreption, job age,salary estimate , and skills are basis str (reg model wont learn, you need nlp)
# min_salary, max_salary to avoid data leakage (since target is avg_salary)
# job age text , and even we made it int, wont be that helpful
# currency captured by country, and avg_salary is in unified currency ($)

In [6]:
df.shape

(716, 381)

In [7]:
# drop target nans always , dont do fillna, or even preproccessing, to avoid leakage, and to combine  fake data with real data in target
# dont train model with nan target valus
df_model=df.dropna(subset='avg_salary').copy() 



In [8]:
df_model.shape

(532, 381)

In [9]:
# df_model.isna().sum()
# df_model.loc[df_model.isna().any(axis=1)]
# # rest nans are in Skills col, X so in preprocessing 


In [10]:
##! split data
X=df_model.drop(columns=drop_cols)
y=df_model['avg_salary']
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2,random_state=42)
X_train.head()

,Hourly,Glassdoor_Estimate,Employer_Provided,Writing skills,SAS,SAP,Relational databases,Data analytics,Project coordination,SDLC,Mandarin,Quantitative analysis,Forecasting,Public speaking,Systems design,Tooling,Organization design,Application development,SharePoint,SolidWorks,C++,Data management,Banking,TestNG,Tableau,Authentication,SVN,Public health,Web analytics,Cassandra,Yardi,IIS,JavaScript,Predictive analytics,Natural language processing,Web accessibility,Quality assurance,Virtualization,Time management,Research & development,Conflict management,Business analysis,Computer science,Embedded software,Debugging,CI/CD,Computer vision,Moving,E-commerce,Regression analysis,Microsoft Access,Oracle,CAD,Business processes,Data mining,System design,Data collection,Salesforce Marketing Cloud,Fiddler,Ansible,D3.js,Multilingual,Pro/ENGINEER,T-SQL,Waterfall,C#,Wireframing,Ontology,Military,XML,Software troubleshooting,SSRS,Market analysis,Epic,OOP,Marketing,CRM software,WAN,Project management,SASS,Risk management,Business requirements,PaaS,HP ALM,Recruiting,Performance tuning,Python,Web development,Program management,Microbiology,Customer retention,Statistical analysis,FedRAMP,Microservices,Release management,Database management,Solution architecture,Controlling experience,Vendor management,Django,Node.js,TypeScript,Elasticsearch,Linux,Distributed systems,Schematics,Databases,Mentoring,Jira,Scala,5G,Wealth management,Help desk,REST,SOX,Spark,Warehouse management system,Google Cloud Platform,Sales,Internet of things,Filing,Logistics,IT,Systems engineering,Microsoft PowerPoint,Financial analysis,.NET,Robotics,Signal processing,QlikView,Continuous improvement,Investment,Data centre experience,Big data,AWS,DevOps,Data structures,Software architecture,Cost management,Scripting,NetSuite,Requirements gathering,Requirements analysis,Data entry,Visual Basic,Load balancing,Microsoft Word,Test automation,MATLAB,Scalability,Financial modeling,Salesforce,Apache Hive,Redshift,Google Suite,Change management,SAP SuccessFactors,Operations management,Data visualization,FISMA,TCP,Confluence,SDKs,Russian,R,Data analysis skills,Deployment,Product development,Computer skills,Salt,Windows,Organisational skills,Management,Kanban,Analysis skills,UI,Performance management,Microsoft Powerpoint,Software testing,Process improvement,English,Figma,Pivot tables,Mechanical engineering,Trello,Medical imaging,Statistical software,IT service management,Hospitality,Cloud development,SaaS,System architecture,VoIP,GAMP,Machine learning,Presentation skills,Microsoft Office,Data warehouse,Disaster recovery,Front-end development,DoD experience,RBAC,ERP systems,System administration,Analytics,Maths,Azure,A/B testing,Japanese,Market research,Clinical development,Database development,Fraud prevention and detection,Bootstrap,TensorFlow,Test-driven development,Operating systems,New Relic,Usability,Math,Software development,Statistics,MongoDB,Contract management,.NET Core,Agile,Operations research,Content development,Marketing mix modeling,Terrafrom,Experimental design,Alteryx,Calculus,ServiceNow,Fanuc,Interviewing,Unity,Pricing,Computer networking,Customer service,Graph databases,FEA,French,DB2,GitHub,Bilingual,Mobile applications,SSO,Docker,Project management software,Data modelling,Microsoft SQL Server,Organizational skills,Clinical research,Supply chain,Taxonomy,HIPAA,Spanish,Cucumber (software testing tool),Telecommunication,Internet of Things,Visual Studio,Financial planning,Crystal Reports,Clinical laboratory experience,Data science,Quality control,EDI,NoSQL,Administrative experience,Biotechnology,Managed care,Team management,Construction,Unreal Engine,Test management tools,S3,Relationship management,Knowledge management,CSS,Root cause analysis,Laboratory information management systems,Metadata,Accounts payable,AutoCAD,Biostatistics,Research,Qualitative research,SCCM,Computer operation,Social listening,Electrical engineering,Kubernetes,Optics,Semantic Web,Barista experience,U

In [11]:
X_train.isna().sum().sum() # good luck that there is no nans, but still we will add imputer (i just want )

np.int64(0)

## Preproccessing

In [12]:
cat_cols=X_train.select_dtypes('str').columns

impute_mode=SimpleImputer(strategy="most_frequent")
ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)

cat_pipe=make_pipeline(impute_mode,ohe)
cat_tuple=['Categorical',cat_pipe,cat_cols]

##! ****************************************************
num_cols=X_train.select_dtypes('number').columns

impute_median=SimpleImputer(strategy='median')
scaler=StandardScaler()

num_pipe=make_pipeline(impute_median,scaler)
num_tuple=['Numarical',num_pipe,num_cols]


In [13]:
# remmeber until now we didnt do any fit, in modeling we will fit with the preprocessor and model at the same time
preprocessor=ColumnTransformer(transformers=[cat_tuple,num_tuple],verbose_feature_names_out=False)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be form

In [14]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error ,root_mean_squared_error

def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = root_mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

# Modeling

## Linear REgression Model

In [ ]:
from sklearn.linear_model import LinearRegression

dummy_lin_reg = LinearRegression()
dummy_lin_reg_pipe=make_pipeline(preprocessor,dummy_lin_reg)
dummy_lin_reg_pipe.fit(X_train,y_train)
dummy_lin_reg_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"spars

In [16]:
print(f"the intercept for dummy linear reg model is {dummy_lin_reg.intercept_}")
print(f"the coefecient for dummy linear reg model is {dummy_lin_reg.coef_}")


the intercept for dummy linear reg model is 130.75845465555656
the coefecient for dummy linear reg model is [ 3.65666533e+01 -2.99217171e+01 -6.64493614e+00 -1.02812874e+01
 -2.00713757e+01 -4.21279615e+00  3.45654593e+01 -2.06152187e+00
  7.31954950e+00  8.34355681e+00 -3.88807324e+00 -4.09476655e-01
  1.46514871e+00  7.66129855e-01 -6.57111491e+00 -1.23012711e-13
 -6.86694447e+00 -6.49530267e-01  3.31166695e+00 -9.56735516e-01
  3.49093259e+00 -6.07793206e+00 -3.11297524e+00 -1.55431223e-14
  3.62217562e+00 -4.28104261e+00 -3.25073302e-13  3.37507799e-14
  2.50580896e+00 -5.17009950e+00  7.68274333e-14 -1.23585141e+01
  9.37028233e-14  6.30606678e-14 -2.19837069e+00  4.47595865e+00
 -5.24679887e+00  5.86546994e-01 -2.03954909e-13 -6.46772002e+00
 -3.14998223e+00  1.01847747e+01 -1.84519067e-13  1.82082374e+00
 -1.17215572e+01 -4.18658297e+00 -8.37108161e-14 -4.13398866e-01
  4.80099764e+00 -2.66082438e+00  3.55271368e-15 -5.28892323e+00
  2.93098879e-14 -3.70771309e+00  5.19780669e+0

In [17]:
evaluate_regression(dummy_lin_reg_pipe,X_train,y_train,X_test,y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 7.677
- MSE = 124.956
- RMSE = 11.178
- R^2 = 0.913

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 25.248
- MSE = 1,832.300
- RMSE = 42.805
- R^2 = -0.228


- Note that these values when we used only new jobs data (/updated_df)
---------------------------------------------------------

1. MAE:
- in training => model is off by 16.78 (~$16.8k)
- in testing => model is off by 47.3 (~$47.3k)
- Overfitting
------------------------------------------------
2. MSE:
- in training => 844
- in testing => 4,604
- model is making very big mistakes in testing data
-----------------------------------------------
3. RMSE:
- in training => 29
- in testing => 67.8
- typical prediction error ~$68k
-----------------------------------------------
4. R2:
- in training => model explains 78% of verience
- in testing => model is worse than predicting the avarage salary for each job
-----------------------------------------------
-----------------------------------------------
### ***the problem is high dimension (368), and too few rows (637)***
- i will fetch more data or better, i will handle old fetched data jobs csv, (back to data understanding and preperation.ipynb)
- i will use skill_cols to extract skills from the new jobs (since there is difference in the job descreption structre)
- new jobs: (SKills : skill1,skill2,skill3,...)
- old jobs: (skills: 3 years in skill1, professional in data cleaning, should be use power pi ,....)
-  so the best way is using the extracted skills from new jobs descreptions, and use it old jobs descreptions
---------------------------------------------------------------------------------------
- Now check the improvemnts when tried it on merged data:
--------------------------------------------------------------
1. MAE:
- in training => model is off by 20.68 (~$20.68k)
- in testing => model is off by 29.3 (~$29.3k)
- better than previous
------------------------------------------------
2. MSE:
- in training => 984.
- in testing => 1,793
- model is making very big mistakes in testing data
-----------------------------------------------
3. RMSE:
- in training => 31.37
- in testing => 42.34
- typical prediction error ~$68k
-----------------------------------------------
4. R2:
- in training => model explains 59% of verience
- in testing => now 25 
----------------------------------------------------------------
## so Grid search enhaned the explained verience in testing, and helped decreasing the overfitting, but still the model verience explaination not good 

In [18]:
dummy_lin_reg_pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[['Categorical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore',
                                                                   sparse_output=False))]),
                                    Index(['seniority', 'Job Category'], dtype='str')],
                                   ['Numarical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     Stan...ler())]),
                                    Index(['

In [19]:
from sklearn.model_selection import GridSearchCV

lin_reg_param_grid = {
    'linearregression__fit_intercept': [True, False],
    'linearregression__positive': [True, False]
}
grid_search_lin_reg=GridSearchCV(estimator= dummy_lin_reg_pipe,param_grid=lin_reg_param_grid,cv=5,scoring='neg_mean_squared_error' )
grid_search_lin_reg.fit(X_train,y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'linearregression__fit_intercept': [True, False], 'linearregression__positive': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candi

In [20]:
evaluate_regression(grid_search_lin_reg,X_train,y_train,X_test,y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 17.339
- MSE = 540.588
- RMSE = 23.251
- R^2 = 0.624

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 21.721
- MSE = 918.523
- RMSE = 30.307
- R^2 = 0.384


## Ridge

In [ ]:
from sklearn.linear_model import Ridge, Lasso
ridge=Ridge()
ridge_pipe=make_pipeline(preprocessor,ridge)
ridge_pipe.fit(X_train,y_train)

evaluate_regression(ridge_pipe,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 7.733
- MSE = 128.245
- RMSE = 11.325
- R^2 = 0.911

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 23.015
- MSE = 1,433.753
- RMSE = 37.865
- R^2 = 0.039


In [23]:
ridge_param_grid = {'ridge__alpha': [0.1, 1, 10, 50, 100, 200, 500]}

ridge_grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)
ridge_grid.fit(X_train, y_train)
evaluate_regression(ridge_grid, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 11.965
- MSE = 283.193
- RMSE = 16.828
- R^2 = 0.803

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 19.054
- MSE = 707.402
- RMSE = 26.597
- R^2 = 0.526


## Lasso

In [ ]:
lasso=Lasso()
lasso_pipe=make_pipeline(preprocessor,lasso)
lasso_pipe.fit(X_train,y_train)

evaluate_regression(lasso_pipe,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 15.494
- MSE = 404.631
- RMSE = 20.115
- R^2 = 0.718

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 19.442
- MSE = 705.295
- RMSE = 26.557
- R^2 = 0.527


In [25]:
lasso_param_grid = {'lasso__alpha': [0.01, 0.1, 1, 5, 10, 50, 100]}

lasso_grid = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)
lasso_grid.fit(X_train, y_train)
print("Best alpha:", lasso_grid.best_params_)
print("\nLasso Tuned:")
evaluate_regression(lasso_grid, X_train, y_train, X_test, y_test)

/home/salah-mohammed/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.013e+03, tolerance: 4.807e+01
  model = cd_fast.enet_coordinate_descent(
/home/salah-mohammed/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.058e+02, tolerance: 4.594e+01
  model = cd_fast.enet_coordinate_descent(
/home/salah-mohammed/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the fe

Best alpha: {'lasso__alpha': 1}

Lasso Tuned:
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 15.494
- MSE = 404.631
- RMSE = 20.115
- R^2 = 0.718

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 19.442
- MSE = 705.295
- RMSE = 26.557
- R^2 = 0.527


In [26]:
df_model.groupby(['seniority', 'Job Category'])['years_exp'].agg(['mean', 'median', 'count'])
# ai engineer senior: 50 year median, a7a
# also ai engineer , and data scientist juniors , are nans, also deep learning senior is nan
#  about ai engineer , and data scientist juniors at most it will be 2 years


mean  median  count
seniority Job Category                           
junior    Data Analyst    3.000000     3.0      1
mid       AI Engineer     4.000000     3.5      8
          Data Analyst    3.093750     3.0     64
          Data Scientist  4.200704     4.0    284
          Deep Learning   2.000000     2.0      1
          ML engineer     5.411765     5.0     17
senior    AI Engineer     4.666667     4.0      3
          Data Analyst    4.818182     5.0     22
          Data Scientist  4.682540     5.0    126
          ML engineer     4.333333     4.0      6

In [27]:
outlier_years=df.loc[df['years_exp']>15]
outlier_years.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Columns: 381 entries, Job Title to skill_num
dtypes: float64(7), int64(354), str(20)
memory usage: 132.0 bytes


In [28]:
# i will use industry standard- years of exp to fill nans:
seniority_years={
    'junior':2,
    'mid':4,
    'senior':7
}
df_model['years_exp'] = df_model.apply(
    lambda row: seniority_years[row['seniority']] 
    if pd.isna(row['years_exp']) 
    else row['years_exp'],
    axis=1
)

In [29]:
df_model.groupby(['seniority', 'Job Category'])['years_exp'].agg(['mean', 'median', 'count'])


mean  median  count
seniority Job Category                           
junior    Data Analyst    3.000000     3.0      1
mid       AI Engineer     4.000000     3.5      8
          Data Analyst    3.093750     3.0     64
          Data Scientist  4.200704     4.0    284
          Deep Learning   2.000000     2.0      1
          ML engineer     5.411765     5.0     17
senior    AI Engineer     4.666667     4.0      3
          Data Analyst    4.818182     5.0     22
          Data Scientist  4.682540     5.0    126
          ML engineer     4.333333     4.0      6

In [30]:
df_model['years_exp'].value_counts()

years_exp
5.0     149
3.0     131
2.0     108
4.0      39
7.0      36
10.0     22
1.0      17
8.0      16
6.0       7
15.0      4
9.0       3
Name: count, dtype: int64

In [31]:
df_model.loc[df_model.index==697]

,Job Title,Company,Location,Salary Estimate,Job Description,Job Age,Unnamed: 0,Rating,Company Name,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Country,Hourly,Glassdoor_Estimate,Employer_Provided,currency,min_salary,max_salary,avg_salary,Skills,cleaned_Job,Writing skills,SAS,SAP,Relational databases,Data analytics,Project coordination,SDLC,Mandarin,Quantitative analysis,Forecasting,Public speaking,Systems design,Tooling,Organization design,Application development,SharePoint,SolidWorks,C++,Data management,Banking,TestNG,Tableau,Authentication,SVN,Public health,Web analytics,Cassandra,Yardi,IIS,JavaScript,Predictive analytics,Natural language processing,Web accessibility,Quality assurance,Virtualization,Time management,Research & development,Conflict management,Business analysis,Computer science,Embedded software,Debugging,CI/CD,Computer vision,Moving,E-commerce,Regression analysis,Microsoft Access,Oracle,CAD,Business processes,Data mining,System design,Data collection,Salesforce Marketing Cloud,Fiddler,Ansible,D3.js,Multilingual,Pro/ENGINEER,T-SQL,Waterfall,C#,Wireframing,Ontology,Military,XML,Software troubleshooting,SSRS,Market analysis,Epic,OOP,Marketing,CRM software,WAN,Project management,SASS,Risk management,Business requirements,PaaS,HP ALM,Recruiting,Performance tuning,Python,Web development,Program management,Microbiology,Customer retention,Statistical analysis,FedRAMP,Microservices,Release management,Database management,Solution architecture,Controlling experience,Vendor management,Django,Node.js,TypeScript,Elasticsearch,Linux,Distributed systems,Schematics,Databases,Mentoring,Jira,Scala,5G,Wealth management,Help desk,REST,SOX,Spark,Warehouse management system,Google Cloud Platform,Sales,Internet of things,Filing,Logistics,IT,Systems engineering,Microsoft PowerPoint,Financial analysis,.NET,Robotics,Signal processing,QlikView,Continuous improvement,Investment,Data centre experience,Big data,AWS,DevOps,Data structures,Software architecture,Cost management,Scripting,NetSuite,Requirements gathering,Requirements analysis,Data entry,Visual Basic,Load balancing,Microsoft Word,Test automation,MATLAB,Scalability,Financial modeling,Salesforce,Apache Hive,Redshift,Google Suite,Change management,SAP SuccessFactors,Operations management,Data visualization,FISMA,TCP,Confluence,SDKs,Russian,R,Data analysis skills,Deployment,Product development,Computer skills,Salt,Windows,Organisational skills,Management,Kanban,Analysis skills,UI,Performance management,Microsoft Powerpoint,Software testing,Process improvement,English,Figma,Pivot tables,Mechanical engineering,Trello,Medical imaging,Statistical software,IT service management,Hospitality,Cloud development,SaaS,System architecture,VoIP,GAMP,Machine learning,Presentation skills,Microsoft Office,Data warehouse,Disaster recovery,Front-end development,DoD experience,RBAC,ERP systems,System administration,Analytics,Maths,Azure,A/B testing,Japanese,Market research,Clinical development,Database development,Fraud prevention and detection,Bootstrap,TensorFlow,Test-driven development,Operating systems,New Relic,Usability,Math,Software development,Statistics,MongoDB,Contract management,.NET Core,Agile,Operations research,Content development,Marketing mix modeling,Terrafrom,Experimental design,Alteryx,Calculus,ServiceNow,Fanuc,Interviewing,Unity,Pricing,Computer networking,Customer service,Graph databases,FEA,French,DB2,GitHub,Bilingual,Mobile applications,SSO,Docker,Project management software,Data modelling,Microsoft SQL Server,Organizational skills,Clinical research,Supply chain,Taxonomy,HIPAA,Spanish,Cucumber (software testing tool),Telecommunication,Internet of Things,Visual Studio,Financial planning,Crystal Reports,Clinical laboratory experience,Data science,Quality control,EDI,NoSQL,Administrative experience,Biotechnology,Managed care,Team management,Construction,Unreal Engine,Test management tools,S3,Relationship management,Knowledge management,CSS,Root cau

## explain the previous models
- shortly : lasso is better than ridge with or without hyperparameter tunning
- MAE: model is off by $23k in training, and $25k in testing
- R2: explains 42% of the avg salary varience
- to explain more vereince, i will try to extract years of experience , but the problem is that not all the new data have years in job descreption as old jobs data,
to handle that i will determine the years based on seniotrity col, (see the mean  years for mid, for senior, for junior, and mid ,.. ) and also use job category since senior in ds hase different years of exp than ai engineer  

## RandomForest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf=RandomForestRegressor(n_estimators=100,random_state=42)
rf_pipeline=make_pipeline(preprocessor,rf)
rf_pipeline.fit(X_train,y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomforestregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"

In [34]:
evaluate_regression(reg=rf_pipeline,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 5.544
- MSE = 72.734
- RMSE = 8.528
- R^2 = 0.949

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 14.965
- MSE = 544.432
- RMSE = 23.333
- R^2 = 0.635


- even better more than lasso 
- but still there is overfitting since r2 in training is 94%, but in test is 63%


In [ ]:
rf_pipeline.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[['Categorical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore',
                                                                   sparse_output=False))]),
                                    Index(['seniority', 'Job Category'], dtype='str')],
                                   ['Numarical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     Stan...ler())]),
                                    Index(['

In [ ]:
rf_grid={
    "randomforestregressor__n_estimators":[100,300],
    "randomforestregressor__max_depth":[10,30],
    "randomforestregressor__min_samples_leaf":[1,2,3,4],
    "randomforestregressor__min_samples_split":[2,5,8]
}
rf_grid=GridSearchCV(estimator=rf_pipeline,
                     param_grid=rf_grid,
                     cv=5,
                     scoring="neg_mean_squared_error",
                     n_jobs=-1,
                     verbose=1
                     )
rf_grid.fit(X_train,y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'randomforestregressor__max_depth': [10, 30], 'randomforestregressor__min_samples_leaf': [1, 2, ...], 'randomforestregressor__min_samples_split': [2, 5, ...], 'randomforestregressor__n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intCo

In [41]:
evaluate_regression(reg=rf_grid,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 5.473
- MSE = 68.988
- RMSE = 8.306
- R^2 = 0.952

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 14.679
- MSE = 514.934
- RMSE = 22.692
- R^2 = 0.655


- not that big deifference

In [47]:
rf_grid.best_params_

{'randomforestregressor__max_depth': 30,
 'randomforestregressor__min_samples_leaf': 1,
 'randomforestregressor__min_samples_split': 2,
 'randomforestregressor__n_estimators': 300}

## XGBOOST

In [ ]:
from xgboost import XGBRegressor

xgb=XGBRegressor(random_state=42,verbosity=0)
xgb_pipe=make_pipeline(preprocessor,xgb)
xgb_pipe.fit(X_train,y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('xgbregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_th

In [49]:
evaluate_regression(reg=xgb_pipe,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)


------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.290
- MSE = 4.167
- RMSE = 2.041
- R^2 = 0.997

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 11.120
- MSE = 413.654
- RMSE = 20.338
- R^2 = 0.723


- significent improvement, but still there is overfitting, since r2 in train 99%, in test 72%
- use tunning to reduce overfitting

In [51]:
xgb_pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[['Categorical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore',
                                                                   sparse_output=False))]),
                                    Index(['seniority', 'Job Category'], dtype='str')],
                                   ['Numarical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     Stan...ler())]),
                                    Index(['

In [56]:
xgb_param_grid = {
    'xgbregressor__n_estimators': [100, 300],
    'xgbregressor__max_depth': [3, 5],
    'xgbregressor__learning_rate': [0.05, 0.1],
    'xgbregressor__subsample': [0.7, 1.0],
    'xgbregressor__colsample_bytree': [0.7, 1.0],
    'xgbregressor__reg_alpha': [0, 1],
    'xgbregressor__reg_lambda': [1, 10]
}
xgb_grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=xgb_param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train)

evaluate_regression(xgb_grid, X_train, y_train, X_test, y_test)

Fitting 5 folds for each of 128 candidates, totalling 640 fits
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 2.601
- MSE = 14.396
- RMSE = 3.794
- R^2 = 0.990

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 12.107
- MSE = 422.420
- RMSE = 20.553
- R^2 = 0.717


the default better & faster

## CONT'd